In [1]:

# imports
import os
import sys
import types
import json

# figure size/format
fig_width = 7
fig_height = 5
fig_format = 'retina'
fig_dpi = 96

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = fig_dpi
  from IPython.display import set_matplotlib_formats
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  pio.renderers.default = "notebook_connected"
except Exception:
  pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass



# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
if r'G:\wjonasreger.github.io\projects\traditional_ngram_language_models':
  os.chdir(r'G:\wjonasreger.github.io\projects\traditional_ngram_language_models')

# reset state
%reset

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v
  
  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define


C:\Users\jonas\AppData\Local\Temp/ipykernel_27476/3555347607.py:20: DeprecationWarning: `set_matplotlib_formats` is deprecated since IPython 7.23, directly use `matplotlib_inline.backend_inline.set_matplotlib_formats()`
  set_matplotlib_formats(fig_format)


{"C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\importlib\\_bootstrap.py": 1630373724.0, "C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\importlib\\_bootstrap_external.py": 1630373724.0, "C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\codecs.py": 1630373724.0, "C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\encodings\\aliases.py": 1630373724.0, "C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\encodings\\__init__.py": 1630373724.0, "C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\encodings\\utf_8.py": 1630373724.0, "C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\encodings\\latin_1.py": 1630373724.0, "C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\abc.py": 1630373724.0, "C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\io.py": 1630373724.0, "C:\\Users\\jonas\\AppData\\Local\\Programs\\Python\\Python39\\lib\\stat.p

In [2]:
#| eval: false
#| echo: false
!pip install torch==1.12.1
!pip install torchdata==0.4.1
!pip install torchtext==0.13.1

In [3]:
import torchtext
import random
import sys
import math
from collections import defaultdict

random.seed(22)

In [4]:
# constants
START = "<START>"   # start-of-sentence token
END = "<END>"    # end-of-sentence-token
UNK = "<UNK>"   # unknown word token

In [5]:
# preprocess function to return preprocessed text data and vocabulary data
def preprocess(data, vocab=None):
    final_data = []
    lowercase = "abcdefghijklmnopqrstuvwxyz"

    for paragraph in data:
        paragraph = [x if x != '<unk>' else UNK for x in paragraph.split()] # replace wikitext-2 <unk> with custom UNK token

        if vocab is not None:
            paragraph = [x if x in vocab else UNK for x in paragraph] # replace unknown words if vocab set is provided

        if paragraph == [] or paragraph.count('=') >= 2: continue # skip empty paragraphs

        sen = []
        prev_punct, prev_quot = False, False

        for word in paragraph:
            if prev_quot: # if word follows a quotation mark
                if word[0] not in lowercase: # simple way to check if punctuation follows quotation to end sentence (susceptible to ending sentences mid-way if contains a quote)
                    final_data.append(sen)
                    sen = []
                    prev_punct, prev_quot = False, False

            if prev_punct: # if word follows punctuation
                if word == '"': # check occurence of quotation marks to help decide to continue or end sentence
                    prev_punct, prev_quot = False, True
                else: # when word follows non-quote punctuations
                    if word[0] not in lowercase: # simple way to check if new sentence is starting (i.e., starts with capitalized letter after punctuation)
                        final_data.append(sen)
                        sen = []
                        prev_punct, prev_quot = False, False

            if word in {'.', '?', '!'}: prev_punct = True
            sen += [word]

        # may need to indent the next two lines later...
        if sen[-1] not in {'.', '?', '!', '"'}: continue # prevent a lot of short sentences (a simple rule)

        final_data.append(sen) # add sentence to final data

    vocab_was_none = vocab is None

    if vocab_was_none:
        vocab = set()

    for i in range(len(final_data)):
        final_data[i] = [START] + final_data[i] + [END] # add start and end tags to sentences

        if vocab_was_none:
            for word in final_data[i]:
                vocab.add(word) # build vocab set

    return final_data, vocab

# function to retreive and preprocess the WikiText-2 dataset
def getDataset():
    dataset = torchtext.datasets.WikiText2(root='.data', split=('train', 'valid'))
    train_dataset, vocab = preprocess(dataset[0]) # preprocessed train and vocab data
    test_dataset, _ = preprocess(dataset[1], vocab) # preprocessed test data with vocab set input

    return train_dataset, test_dataset

train_dataset, test_dataset = getDataset() # instantiate preprocessed test-train data

In [6]:
for sentence in random.sample(train_dataset, 5):
    print(sentence)

['<START>', 'South', 'Korean', 'troops', 'halted', 'the', 'advance', 'of', 'the', 'North', 'Koreans', 'again', 'around', 'the', 'end', 'of', 'the', 'month', 'thanks', 'to', 'increased', 'reinforcements', 'and', 'support', 'closer', 'to', 'the', 'Pusan', 'Perimeter', 'logistics', 'network', '.', '<END>']
['<START>', 'In', 'October', '1970', ',', 'Dylan', 'released', 'New', 'Morning', ',', 'considered', 'a', 'return', 'to', 'form', '.', '<END>']
['<START>', 'Townsend', 'began', 'to', 'record', 'material', 'under', 'the', 'pseudonym', 'Strapping', 'Young', 'Lad', '.', '<END>']
['<START>', 'He', 'was', 'an', 'idealist', 'who', 'disliked', 'the', '<UNK>', 'of', 'politics', ',', 'in', 'fact', '"', 'his', 'innate', 'dislike', 'of', 'politics', 'was', 'something', 'Lady', 'Rosebery', 'always', 'fought', 'against', '.', '"', '<END>']
['<START>', 'Hibari', '@-@', 'kun', '!', '<END>']


In [7]:
# base class defined for language models
class LanguageModel(object):
    def __init__(self, train_data):
        return

    def generateSentence(self):
        raise NotImplementedError("generateSentence is not implemented in each model subclass.")

    def getSentenceLogLikelihood(self, sentence):
        raise NotImplementedError("getSentenceProbability is not implemented in each model subclass.")
        
    def getCorpusPerplexity(self, test_data):
        raise NotImplementedError("getCorpusPerplexity is not implemented in each model subclass.")

    def printSentences(self, n):
        for i in range(n):
            sentence = self.generateSentence()
            sentence_loglik = self.getSentenceLogLikelihood(sentence)
            print('\t[LOG-LIKELIHOOD] ', sentence_loglik , '\n\t[SENTENCE] ', ' '.join(sentence))

In [8]:
def buildModel(ngram_model, train_dataset, test_dataset, sentences=5):
    model = ngram_model(train_dataset)
    print(f'[MODEL TYPE] {ngram_model.__name__}')

    print('\n[RANDOM SENTENCES]')
    model.printSentences(sentences)

    print('\n[CORPUS PERPLEXITIES]')
    print(f'\t[TRAIN DATA] {model.getCorpusPerplexity(train_dataset)}')
    print(f'\t[TEST DATA] {model.getCorpusPerplexity(test_dataset)}')

In [9]:
# a unigram language model class
class UnigramModel(LanguageModel):
    def __init__(self, train_data):
        # generate ngrams
        ngrams = defaultdict(float)
        corpus_size = 0
        for sentence in train_data:
            for word in sentence:
                if word != START:
                    ngrams[word] += 1 # increment ngram count
                    corpus_size += 1 # increment corpus size count
        for vocab in list(ngrams.keys()):
            ngrams[vocab] = ngrams[vocab] / corpus_size # compute estimated ngram probability
        self.ngrams = ngrams

    def generateSentence(self):
        continue_sentence = True
        sentence = [START] # instantiate sentence
        while continue_sentence:
          rand_prob = random.random()
          cum_prob = 0
          for ngram in self.ngrams.keys():
            cum_prob += self.ngrams[ngram]
            if cum_prob >= rand_prob: # find randomly selected ngram
              sentence.append(ngram) # attach ngram to the sentence
              if ngram == END: # check for end of sentence
                continue_sentence = False
              break
        return sentence

    def getSentenceLogLikelihood(self, sentence):
        sentence_loglik = 0
        for word in sentence[1:]: # can ignore first token (i.e., START has probability of 1)
          sentence_loglik += math.log(self.ngrams[word])
        return sentence_loglik
        
    def getCorpusPerplexity(self, test_data):
        corpus_perp = 0
        corpus_size = 0
        for sentence in test_data:
          corpus_perp += -self.getSentenceLogLikelihood(sentence)
          corpus_size += len(sentence) - 1
        return math.exp(corpus_perp / corpus_size) # corpus perplexity

In [10]:
buildModel(UnigramModel, train_dataset, test_dataset, sentences=5)

[MODEL TYPE] UnigramModel

[RANDOM SENTENCES]
	[LOG-LIKELIHOOD]  -475.10553191902574 
	[SENTENCE]  <START> 7 bringing wearing formed , is , 1632 of again might was Brand He <UNK> as 's of . this million Aquila Gray , and for the The cyclone opera MLs her a is specimens was in southeastward with to a Fall on Angelo the addition on <UNK> thought In ; mile stock 's . an . starring , Commission the ) <UNK> released the ( " of cute 18 an , <END>
	[LOG-LIKELIHOOD]  -40.261869916864654 
	[SENTENCE]  <START> critical flexibility guns . the <END>
	[LOG-LIKELIHOOD]  -3.3214585046256566 
	[SENTENCE]  <START> <END>
	[LOG-LIKELIHOOD]  -19.288149711917086 
	[SENTENCE]  <START> episodic , <END>
	[LOG-LIKELIHOOD]  -13.34606467132684 
	[SENTENCE]  <START> opportunity <END>

[CORPUS PERPLEXITIES]


	[TRAIN DATA] 1101.9435880266938
	[TEST DATA] 912.157438593044


In [11]:
# a smoothed unigram language model class
class SmoothedUnigramModel(UnigramModel):
    def __init__(self, train_data):
        ngrams = defaultdict(float)
        corpus_size = 0
        for sen in train_data:
            for word in sen:
                if word != START:
                    ngrams[word] += 1 # increment ngram count
                    corpus_size += 1 # increment corpus size
        for vocab in list(ngrams.keys()):
            ngrams[vocab] = (ngrams[vocab] + 1) / (corpus_size + len(ngrams)) # apply laplace smoothing
        self.ngrams = ngrams

In [12]:
buildModel(SmoothedUnigramModel, train_dataset, test_dataset, sentences=5)

[MODEL TYPE] SmoothedUnigramModel

[RANDOM SENTENCES]
	[LOG-LIKELIHOOD]  -77.77532048070333 
	[SENTENCE]  <START> a give sang although , replaced collector . la as <END>
	[LOG-LIKELIHOOD]  -125.91645198528856 
	[SENTENCE]  <START> the it , . 4th purpose 9 of , the wrote kakapo been has , <UNK> cargo sold States from <END>
	[LOG-LIKELIHOOD]  -396.54035010647124 
	[SENTENCE]  <START> V. One the to Imbert ] the went well , Shakespeare 's to Grandmaster . Wang childish about the ins and ( of a bombed below a opposite the cent would – that was its as 1664 Lord 21 previously . Medal Tech people pay a farmer is as so in fully which used <END>
	[LOG-LIKELIHOOD]  -307.7494657578143 
	[SENTENCE]  <START> end always in Internet , City he closely . ALL in , cause jokes the " pleas 3 and age . his , point , bass dose as and and beast ... so what and show creator , ) early Slow level <UNK> to <END>
	[LOG-LIKELIHOOD]  -83.23978828991501 
	[SENTENCE]  <START> and Song 0600 . vacancy Common secretive o

	[TRAIN DATA] 1103.0243317444856
	[TEST DATA] 914.472450228316


In [13]:
class BigramModel(LanguageModel):
    def __init__(self, train_data):
        ngrams = defaultdict(float)
        for sentence in train_data:
          for i in range(len(sentence)-1):
            word = sentence[i]
            next = sentence[i+1]
            if word not in ngrams.keys():
              ngrams[word] = defaultdict(float) # instantiate set of ngrams for new word w_i
            ngrams[word][next] += 1 # increment ngram count
        for wi_word in list(ngrams.keys()):
          wi_ngrams_set = ngrams[wi_word] # ngram set for word w_i
          wi_ngrams_count = sum(wi_ngrams_set.values()) # ngram count for the set
          for wi_next in list(wi_ngrams_set.keys()):
            ngrams[wi_word][wi_next] = wi_ngrams_set[wi_next] / wi_ngrams_count # ngram probabilities in the set
        self.ngrams = ngrams

    def generateSentence(self):
        continue_sentence = True
        sentence = [START] # instantiate sentence
        while continue_sentence:
          rand_prob = random.random()
          cum_prob = 0
          prev_word = sentence[-1] # previous word in sentence
          for word in self.ngrams[prev_word].keys():
            cum_prob += self.ngrams[prev_word][word]
            if cum_prob >= rand_prob: # find randomly selected ngram
              sentence.append(word) # add ngram to sentece
              if word == END: # search for end of sentence
                continue_sentence = False
              break
        return sentence

    def getSentenceLogLikelihood(self, sentence):
        sentence_prob = 0
        for i in range(len(sentence)-1):
          word = sentence[i]
          next = sentence[i+1]
          try:
            sentence_prob += math.log(self.ngrams[word][next]) # compute log of ngram probability if ngram exists
          except:
            sentence_prob += -float('inf') # compute -inf for log of probability if ngram doesn't exist
        return sentence_prob
        
    def getCorpusPerplexity(self, test_data):
        corpus_perp = 0
        corpus_size = 0
        for sentence in test_data:
          corpus_perp += -self.getSentenceLogLikelihood(sentence) # compute log-likelihood of sentence
          corpus_size += len(sentence) - 1 # increment corpus size
        return math.exp(corpus_perp / corpus_size) # compute perplexity and transform back to original space

In [14]:
buildModel(BigramModel, train_dataset, test_dataset, sentences=5)

[MODEL TYPE] BigramModel

[RANDOM SENTENCES]
	[LOG-LIKELIHOOD]  -56.099616679635375 
	[SENTENCE]  <START> In the Great Charter constituted functional <UNK> and early 20th @-@ metal antimony . <END>
	[LOG-LIKELIHOOD]  -55.816263509674 
	[SENTENCE]  <START> A number one of the winter beak is the performance we are poor . <END>
	[LOG-LIKELIHOOD]  -231.93589365502243 
	[SENTENCE]  <START> Internal Revenue Service Byway is a Norwegian Trunk Railway Museum and a <UNK> International Airport to several American Music from Orsogna , left arm for two paths are due to be possible from the game victories against the key idea incomprehensible to expose the Toronto Students who signed , another . <END>
	[LOG-LIKELIHOOD]  -298.23135275755436 
	[SENTENCE]  <START> He repeated in the island as they could plan for his own display to be a long history " they call to it more commonly recognized but instead he said that is presenting the Hart @-@ 50 @,@ 590 feet ( Sanjeev Kumar and conflicts with 2 – 1 @.@

	[TRAIN DATA] 76.92394608735728
	[TEST DATA] inf


In [15]:
class SmoothedBigramModelAD(BigramModel):
    def __init__(self, train_data):
        # generate ngram probabilities from smoothed unigram model
        self.LPSmoothedUnigram = SmoothedUnigramModel(train_data).ngrams

        # generate ngram counts for bigram model
        ngrams = defaultdict(float)
        for sentence in train_data:
          for w in range(len(sentence)-1):
            word = sentence[w] # current word
            next = sentence[w+1] # next word
            if word not in ngrams.keys():
              ngrams[word] = defaultdict(float) # create ngram set for word
            ngrams[word][next] += 1 # increment ngram count

        # toward computing the discounting factor D
        n1 = 0 # count of ngrams that appear exactly once in train_data
        n2 = 0 # count of ngrams that appear exactly twice in train_data
        for wi_word in list(ngrams.keys()):
            wi_ngrams_set = ngrams[wi_word]
            for wi_next in list(wi_ngrams_set.keys()):
              if ngrams[wi_word][wi_next] == 1:
                n1 += 1 # increment n1 count for D computation
              elif ngrams[wi_word][wi_next] == 2:
                n2 += 1 # increment n2 count for D computation
        D = n1 / (n1 + 2*n2) # compute discounting factor D

        # toward computing ngram types and ngram tokens
        S = defaultdict(float)
        C = defaultdict(float)
        for wi_word in list(ngrams.keys()):
            wi_ngrams_set = ngrams[wi_word]
            S[wi_word] = len(wi_ngrams_set) # number of ngram types
            C[wi_word] = sum(wi_ngrams_set.values()) # number of ngram tokens
        
        # assigning ngrams and smoothing features as attributes for calcProb method
        self.ngrams = ngrams
        self.D = D
        self.S = S
        self.C = C

    def calcProb(self, prev_word, next_word):
        # calculate probability
        try:
          bigram = self.ngrams[prev_word][next_word] # count of bigram token
          t1 = max(bigram - self.D, 0) / self.C[prev_word] # left term of P_AB formula
          t2 = self.D / self.C[prev_word] * self.S[prev_word] * self.LPSmoothedUnigram[next_word] # right term of P_AB formula
          abs_discount_prob = t1 + t2 # probability of next_word given prev_word with absolute discounting
        except:
          abs_discount_prob = 0
        return abs_discount_prob

    def generateSentence(self):
        continue_sentence = True
        sentence = [START] # instantiate sentence
        while continue_sentence:
          rand_prob = random.random()
          cum_prob = 0
          prev_word = sentence[-1]
          for next_word in self.ngrams[prev_word].keys(): # search ngram types/tokens
            cum_prob += self.calcProb(prev_word, next_word) # on-the-go probability calculation
            if cum_prob >= rand_prob: # find randomly selected ngram
              sentence.append(next_word) # add ngram to sentence
              if next_word == END: # search for end of sentence
                continue_sentence = False
              break
        return sentence

    def getSentenceLogLikelihood(self, sentence):
        sentence_prob = 0
        for w in range(len(sentence)-1):
            word = sentence[w]
            next = sentence[w+1]
            sentence_prob += math.log(self.calcProb(word, next))
        return sentence_prob

In [16]:
buildModel(SmoothedBigramModelAD, train_dataset, test_dataset, sentences=5)

[MODEL TYPE] SmoothedBigramModelAD

[RANDOM SENTENCES]
	[LOG-LIKELIHOOD]  -232.5151819640987 
	[SENTENCE]  <START> the Crusader castles in the phenomenon . " to buy 1 , even though the second of television advertisements for the <UNK> football was also , there , who taught by <UNK> and <UNK> drug war starting a set in a new <UNK> it is made further inland , in 2012 ) . <END>
	[LOG-LIKELIHOOD]  -156.3115309602387 
	[SENTENCE]  <START> " . " , Mexico by a fine , legal world No. 13 , although the actress and amino acid plant in the cross through Manga to date with other viewpoints . <END>
	[LOG-LIKELIHOOD]  -15.931381676762696 
	[SENTENCE]  <START> It was 6 . <END>
	[LOG-LIKELIHOOD]  -111.60664837112745 
	[SENTENCE]  <START> After a reaction has been suspected by the right and forbs . " Underneath the Song and Adelaide remained in an additional pressure . <END>
	[LOG-LIKELIHOOD]  -59.42436908068301 
	[SENTENCE]  <START> A small arms of these isotopes such as + 2 mm ) and " <END>

[CORPUS 

	[TRAIN DATA] 98.5581292053259


	[TEST DATA] 272.57201979320354
